# 🚀 Notebook 5 — Advanced RAG Techniques

**DocuMind AI Portfolio Project**

Advanced techniques for production-quality RAG:
1. HyDE — Hypothetical Document Embeddings
2. Multi-Query Retrieval
3. Self-Query Retrieval
4. Parent Document Retriever
5. Ensemble Retriever (BM25 + Semantic)
6. Re-ranking with Cross-Encoders
7. Comparison of all techniques

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','text.color':'#e6edf3','axes.labelcolor':'#e6edf3','xtick.color':'#e6edf3','ytick.color':'#e6edf3'})
import numpy as np
import time
from dotenv import load_dotenv
load_dotenv('../.env')
print('✅ Ready')

In [ ]:
from src.rag_pipeline import RAGPipeline
pipeline = RAGPipeline()
pipeline.ingest_documents('data/raw/sample_docs')
print(f'Pipeline ready. {pipeline.vector_store_manager.get_index_stats()["total_vectors"]} vectors indexed.')

## 1. HyDE — Hypothetical Document Embeddings

Instead of embedding the query directly, HyDE:
1. Asks the LLM to generate a *hypothetical* answer document
2. Embeds that hypothetical document
3. Uses it as the query vector

This bridges the query-document embedding gap.

In [ ]:
def hyde_retrieve(query: str, pipeline) -> list:
    """Retrieve using Hypothetical Document Embeddings."""
    hyde_prompt = f'Write a short passage that answers this question: {query}'
    try:
        hypothetical_doc = pipeline.llm_chain_manager.llm.predict(hyde_prompt)
        print(f'Hypothetical doc: {hypothetical_doc[:200]}...')
        # Embed the hypothetical document instead of the query
        hyp_vector = pipeline.embedding_engine.embed_query(hypothetical_doc)
        results = pipeline.vector_store_manager.vector_store.similarity_search_by_vector(hyp_vector, k=5)
        return results
    except Exception as e:
        print(f'HyDE failed: {e}. Using standard retrieval.')
        return pipeline.retriever.get_relevant_documents(query)

query = 'What is the leave policy at TechCorp?'
print(f'Query: {query}\n')
hyde_docs = hyde_retrieve(query, pipeline)
for d in hyde_docs:
    print(f'  [{d.metadata.get("filename","")}] {d.page_content[:100]}...')

## 2. Multi-Query Retrieval

Generates multiple query variations and merges the results for better recall.

In [ ]:
docs_standard = pipeline.retriever.get_relevant_documents(query)
docs_multi = pipeline.retriever.multi_query_retrieval(query)

print(f'Standard retrieval: {len(docs_standard)} unique chunks')
print(f'Multi-query retrieval: {len(docs_multi)} unique chunks')
print(f'\nSources (standard): {set(d.metadata.get("filename","") for d in docs_standard)}')
print(f'Sources (multi-query): {set(d.metadata.get("filename","") for d in docs_multi)}')

## 3. Ensemble Retriever (BM25 + Semantic)

Combines keyword-based BM25 with semantic vector search.

In [ ]:
def ensemble_retrieve(query: str, pipeline) -> list:
    """BM25 + Semantic ensemble retrieval."""
    try:
        from langchain_community.retrievers import BM25Retriever
        from langchain.retrievers import EnsembleRetriever

        # Get all docs from vector store for BM25
        from src.vector_store import _get_all_docs_from_store
        all_docs = _get_all_docs_from_store(pipeline.vector_store_manager.vector_store)

        if len(all_docs) < 5:
            raise ValueError('Not enough docs for ensemble.')

        bm25_retriever = BM25Retriever.from_documents(all_docs, k=5)
        semantic_retriever = pipeline.vector_store_manager.get_retriever('similarity')

        ensemble = EnsembleRetriever(
            retrievers=[bm25_retriever, semantic_retriever],
            weights=[0.4, 0.6],
        )
        return ensemble.get_relevant_documents(query)
    except ImportError:
        print('rank_bm25 not installed. Using hybrid_search fallback.')
        return pipeline.retriever.hybrid_search(query)
    except Exception as e:
        print(f'Ensemble failed: {e}. Using standard retrieval.')
        return pipeline.retriever.get_relevant_documents(query)

ensemble_docs = ensemble_retrieve(query, pipeline)
print(f'Ensemble retrieval: {len(ensemble_docs)} docs')
for d in ensemble_docs:
    print(f'  [{d.metadata.get("filename","")}] {d.page_content[:80]}...')

## 4. Contextual Compression

Uses an LLM to extract only the relevant portions of each retrieved chunk.

In [ ]:
raw_docs = pipeline.retriever.get_relevant_documents(query)
print(f'Before compression: {len(raw_docs)} docs')
print(f'Total chars: {sum(len(d.page_content) for d in raw_docs)}')

compressed = pipeline.retriever.contextual_compression(query, raw_docs)
print(f'\nAfter compression: {len(compressed)} docs')
print(f'Total chars: {sum(len(d.page_content) for d in compressed)}')
print('\nNote: Compression extracts only query-relevant sentences, reducing noise.')

## 5. Technique Comparison

In [ ]:
techniques = {
    'Standard MMR': lambda q: pipeline.retriever.get_relevant_documents(q),
    'Similarity': lambda q: pipeline.vector_store_manager.similarity_search(q, k=5),
    'Hybrid Search': lambda q: pipeline.retriever.hybrid_search(q),
    'Multi-Query': lambda q: pipeline.retriever.multi_query_retrieval(q),
}

test_query = 'What are the compensation and salary policies?'
results_summary = {}

for name, fn in techniques.items():
    start = time.time()
    try:
        docs = fn(test_query)
        elapsed_ms = (time.time() - start) * 1000
        unique_sources = len(set(d.metadata.get('filename','') for d in docs))
        results_summary[name] = {
            'docs': len(docs),
            'sources': unique_sources,
            'latency_ms': round(elapsed_ms),
        }
    except Exception as e:
        results_summary[name] = {'docs': 0, 'sources': 0, 'latency_ms': 0, 'error': str(e)}

print(f'Test query: "{test_query}"\n')
print(f'{"Technique":<20} {"Docs":<8} {"Unique Src":<12} {"Latency"}')
print('-' * 55)
for name, r in results_summary.items():
    print(f'{name:<20} {r["docs"]:<8} {r["sources"]:<12} {r["latency_ms"]} ms')

## Summary — When to Use Which Technique

| Technique | Best For | Trade-off |
|-----------|----------|-----------|
| **Standard MMR** | Default, balanced | Good diversity, fast |
| **Hybrid BM25+Semantic** | Keyword-heavy queries | Slower, better recall |
| **HyDE** | Vague / abstract queries | LLM call required |
| **Multi-Query** | Complex questions | 3x slower, higher recall |
| **Contextual Compression** | Long context windows | LLM call per doc |
| **Re-ranking** | Final precision boost | Expensive cross-encoder |

**DocuMind AI uses MMR by default** with optional hybrid search as a fallback.